In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset using pandas read_spss
df = pd.read_spss("afrobarometer_release-dataset_nig_r9_en_2023-04-01.sav")

# 2. Map the precise variable names from your metadata trace
COL_STATE = "state"            # Change to "region" if this gives a KeyError
COL_WEIGHT = "withinwt_hh"     # Household design weight variable

# The exact column codes discovered in your file:
COL_TRUST_POLICE = "Q37G"      # Trust police
COL_CORRUPT_POLICE = "Q38E"    # Corruption: Police
COL_UNSAFE_WALK = "Q7A"        # Felt unsafe walking in neighbourhood
COL_FEAR_HOME = "Q7B"          # Feared crime in home

# Updated text percentage calculator
def get_weighted_text_percentage(data, target_col, weight_col, valid_text_responses):
    clean_data = data.dropna(subset=[target_col, weight_col])
    
    # Cast to string and strip whitespace to ensure exact matches
    is_valid = clean_data[target_col].astype(str).str.strip().isin(valid_text_responses)
    
    weighted_numerator = clean_data.loc[is_valid, weight_col].sum()
    weighted_denominator = clean_data[weight_col].sum()
    
    return (weighted_numerator / weighted_denominator) * 100

# --- DIAGNOSTIC RUN ---
print("="*60)
print("RUNNING DIAGNOSTIC RECONCILIATION (TRUE CODES)")
print("="*60)

# Validate Metric 1: Trust police 'Somewhat' or 'A lot' (~15%)
trust_pct = get_weighted_text_percentage(df, COL_TRUST_POLICE, COL_WEIGHT, ["Somewhat", "A lot"])
print(f"Calculated Trust: {trust_pct:.2f}% | Expected: ~15.00%")

# Validate Metric 2: Corruption among 'Most' or 'All' police (~73%)
corrupt_pct = get_weighted_text_percentage(df, COL_CORRUPT_POLICE, COL_WEIGHT, ["Most of them", "All of them"])
print(f"Calculated Corruption: {corrupt_pct:.2f}% | Expected: ~73.00%")

# Validate Metric 3: Felt unsafe walking in neighborhood at least once (~61%)
# Afrobarometer codes: "Just once or twice", "Several times", "Many times", "Always"
unsafe_responses = ["Just once or twice", "Several times", "Many times", "Always"]
unsafe_pct = get_weighted_text_percentage(df, COL_UNSAFE_WALK, COL_WEIGHT, unsafe_responses)
print(f"Calculated Unsafe Walking: {unsafe_pct:.2f}% | Expected: ~61.00%")

print("="*60)

RUNNING DIAGNOSTIC RECONCILIATION (TRUE CODES)
Calculated Trust: 15.00% | Expected: ~15.00%
Calculated Corruption: 73.40% | Expected: ~73.00%
Calculated Unsafe Walking: 61.00% | Expected: ~61.00%


In [2]:
import pandas as pd
import numpy as np

# Suppress the SettingWithCopyWarning using pandas' native configuration option
pd.options.mode.chained_assignment = None

# 1. Load the dataset
df = pd.read_spss("afrobarometer_release-dataset_nig_r9_en_2023-04-01.sav")

# Automatically detect the state/region column regardless of case
potential_state_cols = [col for col in df.columns if col.lower() in ['state', 'region']]
if not potential_state_cols:
    raise ValueError("Could not automatically find a 'state' or 'region' column. Please check your columns.")

COL_STATE = potential_state_cols[0]  # Dynamically selects the correct one (e.g., 'STATE')
COL_WEIGHT = "withinwt_hh"
COL_TRUST_POLICE = "Q37G"
COL_CORRUPT_POLICE = "Q38E"

print(f"-> Successfully detected subnational column: '{COL_STATE}'")

# 2. Map Categorical Responses to a Standardized Numeric Scale (0 to 1)
trust_map = {"Not at all": 0.0, "Just a little": 0.33, "Somewhat": 0.66, "A lot": 1.0}
corrupt_map = {"All of them": 0.0, "Most of them": 0.33, "Some of them": 0.66, "None of them": 1.0}

df['trust_score'] = df[COL_TRUST_POLICE].map(trust_map)
df['corruption_score'] = df[COL_CORRUPT_POLICE].map(corrupt_map)

# Drop missing rows for our calculation components
df_clean = df.dropna(subset=['trust_score', 'corruption_score', COL_WEIGHT]).copy()

# 3. Create a Composite Score at the Individual Level
df_clean['individual_csti'] = (df_clean['trust_score'] + df_clean['corruption_score']) / 2

# 4. Aggregate to State Level Using Weighted Means
def weighted_mean(group):
    return np.average(group['individual_csti'], weights=group[COL_WEIGHT])

# Group by the dynamically detected state variable
# Change your old grouping line to this:
state_csti_dataframe = df_clean.groupby(COL_STATE, observed=False).apply(weighted_mean, include_groups=False).reset_index()
state_csti_dataframe.columns = ['State', 'CSTI']

# Save the final index to a clean CSV
state_csti_dataframe.to_csv("nigeria_state_csti_index.csv", index=False)

print("=" * 60)
print("SUCCESS: State-Level CSTI Index Constructed and Saved!")
print("=" * 60)
print(state_csti_dataframe.head(10))

-> Successfully detected subnational column: 'REGION'
SUCCESS: State-Level CSTI Index Constructed and Saved!
         State      CSTI
0         ABIA  0.120982
1      ADAMAWA  0.372216
2    AKWA IBOM  0.236025
3      ANAMBRA  0.202996
4       BAUCHI  0.305319
5      BAYELSA  0.184027
6        BENUE  0.336582
7        BORNO  0.323718
8  CROSS RIVER  0.186244
9        DELTA  0.180719


In [3]:
print(state_csti_dataframe.head(30))

          State      CSTI
0          ABIA  0.120982
1       ADAMAWA  0.372216
2     AKWA IBOM  0.236025
3       ANAMBRA  0.202996
4        BAUCHI  0.305319
5       BAYELSA  0.184027
6         BENUE  0.336582
7         BORNO  0.323718
8   CROSS RIVER  0.186244
9         DELTA  0.180719
10       EBONYI  0.241887
11          EDO  0.243480
12        EKITI  0.274118
13        ENUGU  0.281164
14    FCT ABUJA  0.403777
15        GOMBE  0.390439
16          IMO  0.165876
17       JIGAWA  0.348227
18       KADUNA  0.266316
19         KANO  0.328286
20      KATSINA  0.328982
21        KEBBI  0.342510
22         KOGI  0.269521
23        KWARA  0.368271
24        LAGOS  0.127092
25     NASARAWA  0.226593
26        NIGER  0.297287
27         OGUN  0.152944
28         ONDO  0.292857
29         OSUN  0.230021
